# Restaurant Sales Intelligence

## Menu Performance, Peak Hours, and Revenue Insights

This notebook presents the business-facing analysis for the project. The goal is to show which products drive revenue, when demand is strongest, where reporting quality is weak, and what actions a restaurant owner could take next.


## Executive Snapshot

- The dataset covers **1,000 orders**, **$275,230 in revenue**, and **8,162 units sold**.
- **Sandwich** is the largest revenue driver, while **Cold coffee** is the highest-volume item.
- **Night** is the strongest selling period by both revenue and average order value.
- **10.7% of transactions** have missing payment type values, which limits checkout-channel reporting.
- The sharp decline in late 2023 should be treated carefully because source coverage is sparse in those months.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

sys.path.append(str(project_root / "src"))

from analysis_pipeline import (  # noqa: E402
    TIME_OF_SALE_ORDER,
    build_data_quality_summary,
    build_summary_tables,
    load_raw_data,
    prepare_clean_dataset,
)

pd.options.display.float_format = "{:,.2f}".format
plt.style.use("ggplot")


In [2]:
raw_df = load_raw_data(project_root / "data" / "raw" / "restaurant_sales.csv")
clean_df = prepare_clean_dataset(raw_df)
quality_df = build_data_quality_summary(clean_df)
tables = build_summary_tables(clean_df)
tables["data_quality_summary"] = quality_df.copy()

print(f"Rows: {len(clean_df):,}")
print(f"Date range: {clean_df['order_date'].min()} to {clean_df['order_date'].max()}")


Rows: 1,000
Date range: 2022-01-04 00:00:00 to 2023-12-03 00:00:00


## 1. Data Reliability Review

Before interpreting performance, it is important to understand which issues were corrected and which limitations remain visible in the cleaned data.

- Mixed date formats were parsed explicitly.
- Missing payment types were preserved as `Unknown` rather than guessed.
- `transaction_amount` matched `item_price * quantity` for every transaction.
- `received_by` only contains `Mr.` and `Mrs.`, so it is documented as a category label rather than a staff identifier.


In [3]:
display(tables["data_quality_summary"])


,issue,affected_rows,status,business_note
0,Mixed date formats,1000,reviewed,403 rows used dd-mm-yyyy and 597 rows used mm/...
1,Missing payment type,107,handled,Missing payment values are retained as Unknown...
2,Transaction amount validation,0,validated,Validated whether transaction_amount equals it...
3,Duplicate order ID check,0,validated,Checked whether order_id values repeat across ...
4,received_by field limitation,1000,documented,received_by contains only honorific labels ['M...
5,Unparsed dates,0,validated,Any rows listed here could not be parsed into ...


## 2. KPI Overview

These metrics summarize the overall scale of the business represented in the dataset.


In [4]:
display(tables["kpi_summary"])


,metric,result
0,Total Revenue,"$275,230"
1,Total Orders,"1,000"
2,Total Quantity Sold,"8,162"
3,Average Order Value,$275.23
4,Average Quantity per Order,8.16
5,Missing Payment Share,10.7%


## 3. Menu Performance

Revenue and volume tell slightly different stories. Higher-priced items can dominate revenue, while strong beverage or snack demand can lead unit volume.


In [5]:
display(tables["top_items_by_revenue"])
display(tables["top_items_by_quantity"])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

revenue_table = tables["top_items_by_revenue"].sort_values("revenue", ascending=True)
quantity_table = tables["top_items_by_quantity"].sort_values("quantity_sold", ascending=True)

axes[0].barh(revenue_table["item_name"], revenue_table["revenue"], color="#0b6e4f")
axes[0].set_title("Top Items by Revenue")
axes[0].set_xlabel("Revenue")

axes[1].barh(quantity_table["item_name"], quantity_table["quantity_sold"], color="#c75c00")
axes[1].set_title("Top Items by Quantity Sold")
axes[1].set_xlabel("Units Sold")

plt.tight_layout()
plt.show()


,rank,item_name,item_type,orders,quantity_sold,revenue,revenue_share_pct,average_order_value
4,1,Sandwich,Fastfood,129,1097,65820,23.91,510.23
2,2,Frankie,Fastfood,139,1150,57500,20.89,413.67
1,3,Cold coffee,Beverages,161,1361,54440,19.78,338.14
5,4,Sugarcane juice,Beverages,153,1278,31950,11.61,208.82
3,5,Panipuri,Fastfood,150,1226,24520,8.91,163.47
0,6,Aalopuri,Fastfood,134,1044,20880,7.59,155.82
6,7,Vadapav,Fastfood,134,1006,20120,7.31,150.15


,rank,item_name,item_type,orders,quantity_sold,revenue,revenue_share_pct,average_order_value,quantity_share_pct
0,1,Cold coffee,Beverages,161,1361,54440,19.78,338.14,16.67
1,2,Sugarcane juice,Beverages,153,1278,31950,11.61,208.82,15.66
2,3,Panipuri,Fastfood,150,1226,24520,8.91,163.47,15.02
3,4,Frankie,Fastfood,139,1150,57500,20.89,413.67,14.09
4,5,Sandwich,Fastfood,129,1097,65820,23.91,510.23,13.44
5,6,Aalopuri,Fastfood,134,1044,20880,7.59,155.82,12.79
6,7,Vadapav,Fastfood,134,1006,20120,7.31,150.15,12.33


C:\Users\sivar\AppData\Local\Temp\ipykernel_41748\1248384697.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Daypart and Monthly Performance

Daypart performance is useful for staffing and prep decisions. Monthly trends provide context on whether sales are stable or uneven across the covered period.


In [6]:
display(tables["sales_by_time_of_sale"])
display(tables["monthly_sales_trend"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

daypart_table = tables["sales_by_time_of_sale"].set_index("time_of_sale").reindex(TIME_OF_SALE_ORDER).reset_index()
axes[0].bar(daypart_table["time_of_sale"], daypart_table["revenue"], color="#4c7cbf")
axes[0].set_title("Sales by Time of Sale")
axes[0].tick_params(axis="x", rotation=20)

monthly_table = tables["monthly_sales_trend"]
axes[1].plot(monthly_table["month"], monthly_table["revenue"], marker="o", color="#0b6e4f")
axes[1].set_title("Monthly Revenue Trend")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


,time_of_sale,orders,quantity_sold,revenue,average_order_value,revenue_share_pct
0,Morning,190,1574,53730,282.79,19.52
1,Afternoon,205,1714,56345,274.85,20.47
2,Evening,201,1540,52355,260.47,19.02
3,Night,205,1759,62075,302.80,22.55
4,Midnight,199,1575,50725,254.90,18.43


,month,orders,quantity_sold,revenue,average_order_value
0,2022-01,21,165,5785,275.48
1,2022-02,27,206,8010,296.67
2,2022-03,23,161,4720,205.22
3,2022-04,65,509,17610,270.92
4,2022-05,84,658,21155,251.85
5,2022-06,66,485,16670,252.58
6,2022-07,66,513,17420,263.94
7,2022-08,85,710,21285,250.41
8,2022-09,82,641,21340,260.24
9,2022-10,76,591,19240,253.16


C:\Users\sivar\AppData\Local\Temp\ipykernel_41748\661329402.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Payment Mix and Category Mix

Payment mix supports checkout analysis, while category mix shows whether the business is more dependent on fast food or beverages.


In [7]:
display(tables["sales_by_payment_type"])
display(tables["item_type_mix"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

payment_table = tables["sales_by_payment_type"]
axes[0].bar(payment_table["payment_type"], payment_table["revenue"], color="#c75c00")
axes[0].set_title("Sales by Payment Type")

category_table = tables["item_type_mix"]
axes[1].bar(category_table["item_type"], category_table["revenue_share_pct"], color="#0b6e4f")
axes[1].set_title("Item Type Revenue Share")
axes[1].set_ylabel("Share (%)")

plt.tight_layout()
plt.show()


,payment_type,orders,quantity_sold,revenue,order_share_pct,average_order_value
0,Cash,476,3943,132840,47.60,279.08
1,Online,417,3291,110595,41.70,265.22
2,Unknown,107,928,31795,10.70,297.15


,item_type,orders,quantity_sold,revenue,revenue_share_pct,quantity_share_pct
1,Fastfood,686,5523,188840,68.61,67.67
0,Beverages,314,2639,86390,31.39,32.33


C:\Users\sivar\AppData\Local\Temp\ipykernel_41748\3500330240.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Best-Performing Category and Daypart Combinations

This view shows where category-specific operations may matter most. It highlights the strongest revenue combinations between item type and selling period.


In [8]:
combo_table = tables["item_type_time_of_sale_performance"].sort_values("revenue", ascending=False)
display(combo_table)


,item_type,time_of_sale,orders,quantity_sold,revenue,average_order_value
8,Fastfood,Night,139,1233,44440,319.71
5,Fastfood,Morning,133,1081,37100,278.95
9,Fastfood,Midnight,145,1135,36680,252.97
7,Fastfood,Evening,139,1022,35880,258.13
6,Fastfood,Afternoon,130,1052,34740,267.23
1,Beverages,Afternoon,75,662,21605,288.07
3,Beverages,Night,66,526,17635,267.20
0,Beverages,Morning,57,493,16630,291.75
2,Beverages,Evening,62,518,16475,265.73
4,Beverages,Midnight,54,440,14045,260.09


## 7. Management Actions

- Protect the top menu drivers by keeping Sandwich, Frankie, and Cold coffee visible in placement, bundles, and promotions.
- Use the strong night performance to guide staffing and prep planning, especially for fast food operations.
- Support afternoon beverage readiness because beverage revenue is relatively strongest in that period.
- Improve payment capture to reduce the unknown payment share and strengthen channel-level reporting.
- Avoid overreacting to the late-2023 drop without confirming whether those months are fully represented in the source data.
